In [0]:
# Databricks SCD2 Snapshot Processing Template

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE_TABLE="udp_qa.bronze.prov_dentaquest_dental_providers"
SILVER_TABLE="udp_qa.silver.dentaquest_fm_service_provider"

df_source=source_df

tracked_cols=["facility_name"]

snapshot_dates=[r.snapshot_date for r in
                df_source.select("snapshot_date").distinct().orderBy("snapshot_date").collect()]

deltaTable=DeltaTable.forName(spark,SILVER_TABLE)

for snapshot in snapshot_dates:

    print(f"Processing {snapshot}")

    df_snapshot=(df_source
        .filter(col("snapshot_date")==snapshot)
        .dropDuplicates(["provider_id"]))

    hash_cols=[coalesce(trim(col(c).cast("string")),lit("")) for c in tracked_cols]

    df_snapshot=df_snapshot.withColumn(
        "record_hash",
        sha2(concat_ws("||",*hash_cols),256)
    )

    df_active=(spark.table(SILVER_TABLE)
        .filter(col("is_current")=="Y"))
    print(f"df_active_count: {df_active.count()}")

    if df_active.limit(1).count()==0:
        (df_snapshot
            .withColumn("effective_start_date",col("snapshot_date"))
            .withColumn("effective_end_date",lit("9999-12-31").cast("date"))
            .withColumn("is_current",lit("Y"))
            .withColumn("created_ts",current_timestamp())
            .withColumn("updated_ts",current_timestamp())
            .write.mode("append")
            .saveAsTable(SILVER_TABLE, mergeSchema=True))
        continue

    df_new=(df_snapshot.alias("s")
            .join(df_active.alias("t"),["provider_id"],"left_anti"))
    print(f"df_new_count: {df_new.count()}")

    df_changed=(df_snapshot.alias("s")
        .join(df_active.alias("t"),["provider_id"])
        .filter(col("s.record_hash")!=col("t.record_hash"))
        .select("s.*"))
    print(f"df_changed_count: {df_changed.count()}")

    df_deleted=(df_active.alias("t")
        .join(df_snapshot.alias("s"),["provider_id"],"left_anti")
        .select("t.provider_id"))
    print(f"df_deleted_count: {df_deleted.count()}")

    if df_changed.limit(1).count()>0:
        (deltaTable.alias("t")
            .merge(df_changed.alias("s"),
                   "t.provider_id=s.provider_id AND t.is_current='Y'")
            .whenMatchedUpdate(set={
                "is_current":"'N'",
                "effective_end_date":"date_sub(s.snapshot_date,1)",
                "updated_ts":"current_timestamp()"
            }).execute())

    if df_deleted.limit(1).count()>0:
        deleted_df=df_deleted.withColumn("snapshot_date",lit(snapshot))
        (deltaTable.alias("t")
            .merge(deleted_df.alias("s"),
                   "t.provider_id=s.provider_id AND t.is_current='Y'")
            .whenMatchedUpdate(set={
                "is_current":"'N'",
                "effective_end_date":"date_sub(s.snapshot_date,1)",
                "updated_ts":"current_timestamp()"
            }).execute())

    insert_df=(df_new.unionByName(df_changed,allowMissingColumns=True)
        .withColumn("effective_start_date",col("snapshot_date"))
        .withColumn("effective_end_date",lit("9999-12-31").cast("date"))
        .withColumn("is_current",lit("Y"))
        .withColumn("created_ts",current_timestamp())
        .withColumn("updated_ts",current_timestamp()))

    if insert_df.limit(1).count()>0:
        insert_df.write.mode("append").saveAsTable(SILVER_TABLE)

print("Completed")
